In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# Get device for training
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([5], device='mps:0')


In [6]:
# Breaking down our nn model into layers
input_image = torch.rand(3, 28, 28)
print(input_image.shape)

torch.Size([3, 28, 28])


In [7]:
# nn.Flatten() converts each 2D 28*28 img into a continous array of 784 pixels
# minibatch dimension (at dim=0) is maintained
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.shape)

torch.Size([3, 784])


In [8]:
# nn.Linear applies a linear transformation on the input using stored weights and biases
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.shape)

torch.Size([3, 20])


In [9]:
# nn.ReLU applied non-linear activations.
# Introducing nonlinearity, helping neural networks learn a wide variety of phenomena.
print(f"Before ReLU: {hidden1}")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1214, -0.4747,  0.0844,  0.0957,  0.5119, -0.2620,  0.3237,  0.2656,
          0.0383,  0.3846, -0.0754, -0.0390,  0.4850,  0.2870, -0.2762,  0.8586,
          0.7925, -0.2760,  0.5076,  0.2629],
        [ 0.0662, -0.2576, -0.1002, -0.2834,  0.7033,  0.3487,  0.3000,  0.2104,
         -0.0692,  0.6950, -0.0665, -0.1798,  0.5569, -0.1274, -0.2377,  0.8831,
          0.6106, -0.2018,  0.4649,  0.2371],
        [-0.2341, -0.3710, -0.1717, -0.0665,  0.2259,  0.2734,  0.4802,  0.1072,
          0.0876,  0.9538, -0.0579, -0.2057,  0.4450,  0.0157, -0.2145,  0.9955,
          0.6878, -0.3073,  0.4109,  0.1407]], grad_fn=<AddmmBackward0>)
After ReLU: tensor([[0.0000, 0.0000, 0.0844, 0.0957, 0.5119, 0.0000, 0.3237, 0.2656, 0.0383,
         0.3846, 0.0000, 0.0000, 0.4850, 0.2870, 0.0000, 0.8586, 0.7925, 0.0000,
         0.5076, 0.2629],
        [0.0662, 0.0000, 0.0000, 0.0000, 0.7033, 0.3487, 0.3000, 0.2104, 0.0000,
         0.6950, 0.0000, 0.0000, 0.5569, 0.0000, 0.0000

In [11]:
# nn.Sequential is an ordered container of modules.
# Data processed the same order as defined.
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10),
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
print(logits.shape)

torch.Size([3, 10])


In [13]:
# nn.Softmax scales logits from [-inf, +inf] to [0, 1]
# representing the model's predicted probabilities for each class
# `dim` indicates the dimension along which the values must sum to 1
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab.shape)
print(pred_probab)

torch.Size([3, 10])
tensor([[0.1434, 0.0870, 0.0979, 0.1144, 0.0935, 0.0859, 0.0910, 0.0963, 0.0879,
         0.1028],
        [0.1438, 0.0798, 0.1048, 0.1077, 0.0954, 0.0863, 0.0755, 0.1023, 0.0887,
         0.1157],
        [0.1594, 0.0776, 0.1016, 0.1149, 0.0968, 0.0902, 0.0787, 0.0871, 0.0883,
         0.1054]], grad_fn=<SoftmaxBackward0>)


In [14]:
print(f"Model structure: {model}")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]}")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[ 0.0074, -0.0059, -0.0313,  ...,  0.0239,  0.0280,  0.0330],
        [-0.0266, -0.0030, -0.0086,  ..., -0.0220, -0.0145, -0.0341]],
       device='mps:0', grad_fn=<SliceBackward0>)
Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([ 0.0105, -0.0280], device='mps:0', grad_fn=<SliceBackward0>)
Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0394,  0.0065,  0.0227,  ..., -0.0441,  0.0094,  0.0035],
        [ 0.0359,  0.0047,  0.0375,  ..., -0.0411, -0.0415, -0.0004]],
       device='mps:0', grad_fn=<SliceBackward0